# 7장 — 커지다 멈추는 성장: S자 곡선과 지배 루프

웹 챕터(7장)와 **같은 모형, 같은 수치**를 파이썬으로 계산합니다:
로지스틱 성장의 변곡점(K/2)과 최대 가입 속도(rK/4), 시그모이드와의 관계, 초기 숫자로 지수 곡선을 그리면 생기는 일.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch7')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 로지스틱 성장

$$\frac{dS}{dt} = r\,S\left(1 - \frac{S}{K}\right) \quad\Longrightarrow\quad S(t) = \frac{K}{1 + \frac{K - S_0}{S_0}\, e^{-rt}}$$

처음 사용자 10명, 수용 한계 K = 1,000명, 성장률 r = 0.5/월. 정확한 답과 오일러법(DT = 1/16, 5장)을 비교합니다.

**그림으로 읽기**(웹 7장 1절): 인과지도에는 고리가 둘입니다. **입소문(R)**: 사용자 수 ↑ → (+) 신규 가입 ↑ → (+) 사용자 수 ↑(− 0개).
**포화(B)**: 사용자 수 ↑ → (−) 남은 여지 ↓ → (+) 신규 가입 ↓ → (+) 사용자 수가 덜 는다(− 1개). 두 고리가 '신규 가입'을 반대 방향으로 미는
줄다리기이고, 지금 이기는 쪽이 지배 루프입니다(2절).

In [ ]:
import math

S0, K, r = 10, 1000, 0.5

def exact(t, r=r, K=K):
    return K / (1 + (K - S0) / S0 * math.exp(-r * t))

def euler(dt=1 / 16, T=36):
    S, out = S0, [S0]
    for _ in range(round(T / dt)):
        S += r * S * (1 - S / K) * dt
        out.append(S)
    return out

t_star = math.log((K - S0) / S0) / r
print(f'변곡점 t* = ln(99)/{r} = {t_star:.2f}개월, 그때 사용자 {exact(t_star):.1f}명, 최대 가입 속도 rK/4 = {r * K / 4:.0f}명/월')
for m in [3, 6, 9, 12, 18, 24, 36]:
    print(f'{m:2d}개월  정확 {exact(m):7.1f}명   오일러(DT=1/16) {euler()[m * 16]:7.1f}명')

# 웹 실험실(LogisticLab)·본문 수치 자가검증
assert round(t_star, 2) == 9.19 and abs(exact(t_star) - 500) < 1e-9
assert r * K / 4 == 125
assert round(exact(24)) == 999
err = max(abs(euler()[i] - exact(i / 16)) for i in range(36 * 16 + 1))
print(f'오일러(DT = 1/16)와 정확한 답의 최대 차이: {err:.1f}명 (K의 {err / K * 100:.1f}%)')
assert err < 15
print('✅ 변곡점 9.19개월(500명) · 최대 가입 속도 125명/월')

## 2. 변곡점에서 지배 루프가 바뀐다

가입 속도 $r S (1 - S/K)$ 를 사용자 수 S의 함수로 보면 $S = K/2$ 에서 최대인 포물선입니다.
그 전에는 S가 늘수록 가입이 늘고(강화 루프가 이김), 그 뒤에는 S가 늘수록 가입이 줄어요(균형 루프가 이김).

In [ ]:
flow = lambda S: r * S * (1 - S / K)
for S in [10, 100, 250, 500, 750, 900, 990]:
    slope = r * (1 - 2 * S / K)       # 가입 속도가 S에 따라 늘어나는 정도(미분)
    who = '강화 루프가 이김' if slope > 0 else ('균형 루프가 이김' if slope < 0 else '변곡점')
    print(f'S = {S:4d}명: 가입 속도 {flow(S):6.1f}명/월,  S가 늘 때 가입 속도의 변화 {slope:+.2f}  → {who}')
assert flow(500) == max(flow(s) for s in range(0, 1001))
print('✅ 가입 속도는 S = K/2 에서 최대, 그 앞뒤로 지배 루프가 바뀐다')

## 3. 로지스틱 = 시그모이드

2부 7장의 시그모이드 $\sigma(x) = 1/(1 + e^{-x})$ 로 쓰면 $S(t) = K\,\sigma\big(r(t - t^*)\big)$.

In [ ]:
sigma = lambda x: 1 / (1 + math.exp(-x))
diff = max(abs(exact(t) - K * sigma(r * (t - t_star))) for t in [i * 0.25 for i in range(145)])
print(f'로지스틱 정확한 답과 K·σ(r(t − t*))의 최대 차이: {diff:.2e}')
assert diff < 1e-9
print('✅ S자 성장 곡선 = 시그모이드를 세로 K배·가로 1/r배 늘이고 t*만큼 옮긴 것')

## 4. 처음 3개월로 미래를 재면

처음 3개월 숫자로 '매달 몇 배' 지수 곡선을 맞춰 2년 뒤를 예측해 봅시다.

In [ ]:
g = (exact(3) / S0) ** (1 / 3)
pred24 = S0 * g ** 24
print(f'3개월 뒤 {exact(3):.1f}명 → 매달 {g:.3f}배로 맞춤 → 24개월 예측 {pred24:,.0f}명 (실제 {exact(24):.0f}명)')
assert round(exact(3)) == 43 and round(g, 2) == 1.63
assert 1.2e6 < pred24 < 1.3e6       # 약 124만 명
print('✅ S자의 앞부분은 지수와 구별이 안 된다. 한계(균형 루프)를 빼면 터무니없는 예측이 나온다')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ts = [i * 0.25 for i in range(145)]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6.4), sharex=True)
for rr, c in [(0.3, '#6ee7b7'), (0.5, '#059669'), (0.8, '#065f46')]:
    ax1.plot(ts, [exact(t, rr) for t in ts], color=c, lw=2.2, label=f'r = {rr}')
    ax2.plot(ts, [rr * exact(t, rr) * (1 - exact(t, rr) / K) for t in ts], color=c, lw=2)
ax1.plot(ts[:60], [S0 * g ** t for t in ts[:60]], '--', color='#f43f5e', label='처음 3개월 기세대로(지수)')
ax1.axhline(K, ls='--', color='#2563eb', lw=1)
ax1.axvline(t_star, ls=':', color='#d97706')
ax1.set_ylim(0, 1150); ax1.set_ylabel('사용자 수(명)'); ax1.set_title('S자 성장: 변곡점(점선) 앞은 강화, 뒤는 균형이 이긴다'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.axvline(t_star, ls=':', color='#d97706')
ax2.set_xlabel('개월'); ax2.set_ylabel('가입 속도(명/월)'); ax2.set_title('가입 속도는 변곡점에서 최대 rK/4'); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 오늘의 발견

- **로지스틱 성장** = 강화 루프 + 포화하는 균형 루프: $dS/dt = rS(1 - S/K)$.
- **변곡점**은 $S = K/2$, 시각 $t^* = \ln((K - S_0)/S_0)/r$ (예: 9.19개월). 그때 가입 속도가 최대 $rK/4$ (125명/월).
- 변곡점 앞은 강화 루프, 뒤는 균형 루프가 **지배 루프**. 가입 정점 ≠ 사용자 정점.
- 답은 **시그모이드**: $S(t) = K\sigma(r(t - t^*))$ (2부 7장·6부 4장의 그 함수).
- S자의 앞부분으로 지수 곡선을 그리면 크게 과대예측한다(2년 뒤 약 124만 명 vs 실제 1,000명).

다음 장(8장 공은 어디서 멈출까)에서 평형과 안정성을 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**